[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/D-Barradas/Accelerated-Data-Science-with-RAPIDS/blob/main/part4/4-02_HPO_RayTune_Fractional_GPU_RAPIDS.ipynb)

# Part 4.2: High-Throughput HPO using Ray Tune and Fractional GPU Allocation with RAPIDS & PyTorch

**Series:** Accelerated Data Science with RAPIDS — Part 4: Hyperparameter Optimization

## What you will learn

Google Colab typically gives you a single GPU (T4 or L4). Running hyperparameter optimization (HPO) trials one at a time on that single GPU wastes most of its compute and memory capacity, because a small-to-medium PyTorch model rarely saturates a modern GPU on its own. This notebook shows how to:

1. Generate and feature-engineer a large (500k+ row) synthetic time-series dataset entirely on the GPU using **cuDF**, so preprocessing never becomes the bottleneck.
2. Move that data from cuDF into PyTorch-compatible tensors efficiently.
3. Define a small, configurable 1D-CNN / LSTM forecasting model.
4. Use **Ray Tune** with **fractional GPU allocation** (`gpu=0.33`) to pack **three concurrent PyTorch trials onto one physical GPU**, and use the **ASHA scheduler** to kill unpromising trials early so the GPU time we save is reinvested in more promising configurations.

By the end, you'll understand *why* fractional GPU scheduling + early stopping is one of the highest-leverage techniques for doing serious HPO on a single-GPU budget (Colab, a laptop, or a single cloud instance).

## 1. Setup & Environment Verification

Run the cell below once per Colab session. We pin `cudf-cu12` from NVIDIA's package index (RAPIDS wheels are not on the default PyPI index) and install `ray[tune]` for the HPO framework. `torch` ships pre-installed on Colab GPU runtimes with CUDA already wired up, but we install/upgrade it explicitly so this notebook is reproducible outside Colab too.

> **Tip:** If `cudf`/`ray` fail to import right after installing, use **Runtime -> Restart runtime** (not "restart and run all") and re-run from the top — this clears any partially-initialized CUDA context left by the installer.

In [ ]:
# !pip -q install torch torchvision scikit-learn matplotlib pandas
!pip -q install "ray[tune]"

# RAPIDS for CUDA 12 (best effort for Colab).
# !pip -q install --extra-index-url=https://pypi.nvidia.com cudf-cu12 cuml-cu12 cupy-cuda12x || true

In [ ]:
# Verify the GPU, PyTorch/CUDA, and RAPIDS/Ray versions before doing any real work.
# Failing fast here saves time versus discovering a broken environment mid-training.
import subprocess

import cudf
import ray
import torch

print("=" * 60)
print("GPU (nvidia-smi)")
print("=" * 60)
try:
    smi = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
        capture_output=True, text=True, check=True,
    )
    print(smi.stdout)
except (FileNotFoundError, subprocess.CalledProcessError) as exc:
    print(f"nvidia-smi unavailable: {exc}. Make sure the Colab runtime type is set to GPU.")

print("=" * 60)
print("Library versions")
print("=" * 60)
print(f"torch             : {torch.__version__}")
print(f"torch.cuda avail  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"torch CUDA device : {torch.cuda.get_device_name(0)}")
    print(f"torch CUDA version: {torch.version.cuda}")
print(f"cudf              : {cudf.__version__}")
print(f"ray               : {ray.__version__}")

assert torch.cuda.is_available(), (
    "No CUDA device found. In Colab: Runtime -> Change runtime type -> Hardware accelerator -> GPU (T4/L4)."
)

## 2. Accelerated Data Pipeline with cuDF

We simulate a realistic **multi-entity sensor/financial time series**: many independent entities (e.g. machines, tickers, IoT sensors), each sampled at regular intervals, each producing a noisy signal with its own baseline level, trend, and seasonality.

All feature engineering — rolling statistics, lag features, and grouped/categorical aggregates — is done with **cuDF**, so a 500k-row dataset that would take tens of seconds with pandas `groupby().rolling()` finishes in a fraction of a second on the GPU. This matters because in a real HPO workflow you often re-run preprocessing many times while iterating on features; a slow CPU pipeline directly taxes your GPU-trial throughput budget.

In [ ]:
# Core imports used throughout the rest of the notebook.
import numpy as np
import cupy as cp
import matplotlib.pyplot as plt

import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torch.utils.dlpack import from_dlpack

from ray import tune, train
from ray.tune.schedulers import ASHAScheduler

In [ ]:
# Generate a synthetic multi-entity time-series dataset directly as a cuDF DataFrame.
# 200 entities x 2,500 timesteps = 500,000 rows.
N_ENTITIES = 200
N_TIMESTEPS = 2_500
SEED = 42

cp.random.seed(SEED)

entity_ids = cp.repeat(cp.arange(N_ENTITIES, dtype=cp.int32), N_TIMESTEPS)
timesteps = cp.tile(cp.arange(N_TIMESTEPS, dtype=cp.int32), N_ENTITIES)

# Each entity has its own random baseline level and slow drift/trend.
entity_baseline = cp.random.normal(loc=0.0, scale=5.0, size=N_ENTITIES).repeat(N_TIMESTEPS)
entity_trend = cp.random.normal(loc=0.0, scale=0.01, size=N_ENTITIES).repeat(N_TIMESTEPS)

# Signal = baseline + slow trend + seasonality + noise.
seasonality = 3.0 * cp.sin(2 * cp.pi * timesteps / 50.0)
noise = cp.random.normal(loc=0.0, scale=1.0, size=N_ENTITIES * N_TIMESTEPS)
value = entity_baseline + entity_trend * timesteps + seasonality + noise

gdf = cudf.DataFrame({
    "entity_id": entity_ids,
    "timestamp": timesteps,
    "value": value.astype(cp.float32),
})

# cuDF requires data sorted-by-group for correct rolling/lag semantics.
gdf = gdf.sort_values(["entity_id", "timestamp"]).reset_index(drop=True)

print(f"Rows: {len(gdf):,}")
gdf.head()

In [ ]:
# Feature engineering on GPU: rolling stats, lag features, and grouped/categorical aggregates.
# Everything here runs as a groupby-rolling / groupby-transform directly on the GPU with cuDF.
ROLL_WINDOW = 10

grouped = gdf.groupby("entity_id", sort=False)

# Rolling window aggregations: per-entity moving average and standard deviation of the signal.
gdf["roll_mean"] = grouped["value"].rolling(ROLL_WINDOW, min_periods=1).mean().reset_index(drop=True)
gdf["roll_std"] = grouped["value"].rolling(ROLL_WINDOW, min_periods=1).std().reset_index(drop=True)
gdf["roll_std"] = gdf["roll_std"].fillna(0.0)

# Lag features: value at t-1, t-2, t-3 (per entity, no cross-entity leakage).
for lag in (1, 2, 3):
    gdf[f"lag_{lag}"] = grouped["value"].shift(lag).reset_index(drop=True)
gdf[["lag_1", "lag_2", "lag_3"]] = gdf[["lag_1", "lag_2", "lag_3"]].bfill()

# Grouped/categorical feature: overall per-entity mean, broadcast back onto every row.
# This is the classic "categorical -> aggregate" feature pattern used in tabular ML pipelines.
gdf["entity_mean"] = grouped["value"].transform("mean")

# Forecasting target: next-step value (what the model predicts from the trailing window).
gdf["target"] = grouped["value"].shift(-1).reset_index(drop=True)

# Drop the last timestep per entity (no target available) and any residual NaNs.
gdf = gdf.dropna().reset_index(drop=True)

print(f"Rows after feature engineering: {len(gdf):,}")
gdf.head()

### From tabular rows to fixed-length sequences

The forecasting model needs a contiguous trailing window of `SEQ_LEN` timesteps per training example. We build these windows with vectorized GPU array indexing (broadcasted fancy indexing) instead of a Python-level loop, so this step scales cleanly to the full 500k-row dataset.

In [ ]:
# Build fixed-length sequences (sliding windows) for the forecasting model.
#
#   1. Number rows within each entity (row_id = 0..n_i-1).
#   2. Keep rows whose row_id >= SEQ_LEN - 1 (enough history exists).
#   3. Gather SEQ_LEN-length windows via broadcasted fancy indexing on the feature matrix.
SEQ_LEN = 24
FEATURE_COLS = ["value", "roll_mean", "roll_std", "lag_1", "lag_2", "lag_3", "entity_mean"]

gdf["row_id"] = grouped.cumcount()

# Feature matrix and targets as CuPy arrays (zero-copy view from cuDF).
feature_matrix = gdf[FEATURE_COLS].to_cupy(dtype=cp.float32)
targets = gdf["target"].to_cupy(dtype=cp.float32)
row_id = gdf["row_id"].to_cupy(dtype=cp.int32)

valid_mask = row_id >= (SEQ_LEN - 1)
end_indices = cp.nonzero(valid_mask)[0]  # global row index of the *last* step in each window

# window_indices[i] = [end_indices[i] - SEQ_LEN + 1, ..., end_indices[i]]
offsets = cp.arange(SEQ_LEN, dtype=cp.int32) - (SEQ_LEN - 1)
window_indices = end_indices[:, None] + offsets[None, :]

X = feature_matrix[window_indices]          # shape: (n_sequences, SEQ_LEN, n_features)
y = targets[end_indices]                    # shape: (n_sequences,)

print(f"Sequence tensor X: {X.shape}, target vector y: {y.shape}")

In [ ]:
# Normalize features (zero mean / unit variance) and hold out a validation split.
# Normalization statistics are computed on the *training* split only, to avoid leakage.
N = X.shape[0]
n_val = int(0.2 * N)
perm = cp.random.permutation(N)
train_idx, val_idx = perm[n_val:], perm[:n_val]

X_train, X_val = X[train_idx], X[val_idx]
y_train, y_val = y[train_idx], y[val_idx]

feat_mean = X_train.reshape(-1, X_train.shape[-1]).mean(axis=0)
feat_std = X_train.reshape(-1, X_train.shape[-1]).std(axis=0) + 1e-6
X_train = (X_train - feat_mean) / feat_std
X_val = (X_val - feat_mean) / feat_std

target_mean, target_std = y_train.mean(), y_train.std() + 1e-6
y_train = (y_train - target_mean) / target_std
y_val = (y_val - target_mean) / target_std

print(f"Train sequences: {X_train.shape[0]:,} | Val sequences: {X_val.shape[0]:,}")

### Handing data to PyTorch

We convert the CuPy arrays to host NumPy arrays (via a zero-copy DLPack GPU tensor) before building `TensorDataset`/`DataLoader` objects. We move to host memory at this point because **Ray Tune trials run in separate worker processes**: a `DataLoader` holding live CUDA tensors cannot be pickled and shared across process boundaries, so each trial reconstructs its own `DataLoader` from shared NumPy arrays (Section 4) and moves batches to its assigned GPU fraction itself.

In [ ]:
def cupy_to_numpy(arr: cp.ndarray) -> np.ndarray:
    """Convert a CuPy array to a host NumPy array via a zero-copy DLPack GPU tensor."""
    return from_dlpack(arr.toDlpack()).cpu().numpy()


X_train_np, y_train_np = cupy_to_numpy(X_train), cupy_to_numpy(y_train)
X_val_np, y_val_np = cupy_to_numpy(X_val), cupy_to_numpy(y_val)

# Demonstrate the direct cuDF -> PyTorch path: build TensorDataset/DataLoader objects.
train_dataset = TensorDataset(torch.from_numpy(X_train_np), torch.from_numpy(y_train_np))
val_dataset = TensorDataset(torch.from_numpy(X_val_np), torch.from_numpy(y_val_np))

demo_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
xb, yb = next(iter(demo_loader))
print(f"Example batch: X={tuple(xb.shape)}, y={tuple(yb.shape)}, dtype={xb.dtype}")

## 3. PyTorch Model Definition

We define a single modular `SequenceForecastNet` that can act as **either a 1D-CNN or an LSTM** encoder over the trailing `SEQ_LEN`-step window, followed by a small MLP regression head. Which encoder is used, how wide/deep it is, and its dropout rate are all constructor arguments — this lets Ray Tune treat the architecture family itself as a tunable hyperparameter, not just its size.

In [ ]:
class SequenceForecastNet(nn.Module):
    """Configurable 1D-CNN / LSTM encoder for single-step time-series forecasting.

    Parameters
    ----------
    n_features : number of input features per timestep.
    hidden_size : width of the encoder (CNN channels or LSTM hidden units).
    num_layers : depth of the encoder.
    dropout : dropout probability applied inside the encoder / before the regression head.
    model_type : "lstm" or "cnn" - selects the sequence encoder architecture.
    """

    def __init__(self, n_features: int, hidden_size: int = 64, num_layers: int = 2,
                 dropout: float = 0.2, model_type: str = "lstm"):
        super().__init__()
        self.model_type = model_type

        if model_type == "lstm":
            self.encoder = nn.LSTM(
                input_size=n_features,
                hidden_size=hidden_size,
                num_layers=num_layers,
                batch_first=True,
                dropout=dropout if num_layers > 1 else 0.0,
            )
        elif model_type == "cnn":
            channels = [n_features] + [hidden_size] * num_layers
            conv_layers = []
            for in_ch, out_ch in zip(channels[:-1], channels[1:]):
                conv_layers += [
                    nn.Conv1d(in_ch, out_ch, kernel_size=3, padding=1),
                    nn.ReLU(),
                    nn.Dropout(dropout),
                ]
            self.encoder = nn.Sequential(*conv_layers)
        else:
            raise ValueError(f"Unknown model_type: {model_type!r} (expected 'lstm' or 'cnn')")

        self.head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (batch, seq_len, n_features)
        if self.model_type == "lstm":
            _, (h_n, _) = self.encoder(x)
            encoded = h_n[-1]                          # last layer's final hidden state
        else:
            encoded = self.encoder(x.transpose(1, 2))   # -> (batch, channels, seq_len)
            encoded = encoded.mean(dim=-1)               # global average pool over time
        return self.head(encoded).squeeze(-1)

In [ ]:
# Quick sanity check: one forward pass with random weights on CPU, for both architectures.
for _model_type in ("lstm", "cnn"):
    _sanity_model = SequenceForecastNet(n_features=len(FEATURE_COLS), model_type=_model_type)
    _sanity_out = _sanity_model(torch.from_numpy(X_train_np[:8]))
    assert _sanity_out.shape == (8,), f"Unexpected output shape: {_sanity_out.shape}"
    print(f"{_model_type} forward pass OK: output shape {tuple(_sanity_out.shape)}")

## 4. Ray Tune Integration & Fractional GPU Allocation

**Why fractional GPUs?** A single trial of this model uses only a small fraction of a T4/L4's compute and a few hundred MB of memory — running one trial at a time would leave most of the GPU idle. Ray Tune lets you declare *fractional* GPU requirements per trial (e.g. `gpu=0.33`); Ray then **co-schedules multiple trials on the same physical GPU**, each with its own CUDA context, as long as the fractions requested sum to at most 1 per GPU. With `gpu=0.33` we can run **3 trials concurrently** on Colab's single GPU — roughly a 3x increase in HPO throughput for free.

**Why ASHA?** Successive-halving schedulers like ASHA monitor each trial's intermediate metrics and terminate the worst-performing trials early, before they consume their full epoch budget. Combined with fractional GPUs, this means the GPU slots freed by killed trials are immediately reused by new configurations — maximizing the number of configurations explored per unit of wall-clock time.

In [ ]:
# Initialize a local Ray cluster sized for Colab's resource limits.
# `ignore_reinit_error=True` lets you re-run this cell without restarting the kernel.
import multiprocessing

ray.shutdown()  # ensure a clean slate if this cell is re-run
ray.init(
    num_cpus=multiprocessing.cpu_count(),
    num_gpus=1 if torch.cuda.is_available() else 0,
    ignore_reinit_error=True,
    log_to_driver=False,  # keep trial logs from flooding the notebook output
)
print(ray.cluster_resources())

In [ ]:
# The Ray Tune objective. `tune.with_parameters` (used below) places the NumPy arrays into
# Ray's object store once and hands each trial a reference, avoiding N re-serializations of
# the dataset for N trials.
def train_forecast(config: dict, X_train, y_train, X_val, y_val):
    """Train one SequenceForecastNet configuration and report metrics each epoch."""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = SequenceForecastNet(
        n_features=X_train.shape[-1],
        hidden_size=config["hidden_size"],
        num_layers=config["num_layers"],
        dropout=config["dropout"],
        model_type=config["model_type"],
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=config["lr"])
    criterion = nn.MSELoss()

    train_loader = DataLoader(
        TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train)),
        batch_size=config["batch_size"], shuffle=True, drop_last=True,
    )
    val_loader = DataLoader(
        TensorDataset(torch.from_numpy(X_val), torch.from_numpy(y_val)),
        batch_size=config["batch_size"], shuffle=False,
    )

    for epoch in range(config.get("max_epochs", 15)):
        model.train()
        train_loss_total, train_batches = 0.0, 0
        try:
            for xb, yb in train_loader:
                xb, yb = xb.to(device), yb.to(device)
                optimizer.zero_grad()
                loss = criterion(model(xb), yb)
                loss.backward()
                optimizer.step()
                train_loss_total += loss.item()
                train_batches += 1
        except RuntimeError as exc:
            # With several trials sharing one GPU, a memory spike can exhaust a trial's
            # fractional allocation. Report a poor score so ASHA prunes this trial instead
            # of the whole tuning run crashing.
            if "out of memory" not in str(exc).lower():
                raise
            torch.cuda.empty_cache()
            train.report({"val_loss": float("inf"), "train_loss": float("inf"),
                          "epoch": epoch, "oom": True})
            return

        model.eval()
        val_loss_total, val_batches = 0.0, 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                val_loss_total += criterion(model(xb), yb).item()
                val_batches += 1

        train.report({
            "train_loss": train_loss_total / max(train_batches, 1),
            "val_loss": val_loss_total / max(val_batches, 1),
            "epoch": epoch,
        })

In [ ]:
# Hyperparameter search space.
search_space = {
    "lr": tune.loguniform(1e-4, 1e-2),
    "batch_size": tune.choice([64, 128, 256]),
    "dropout": tune.uniform(0.0, 0.5),
    "hidden_size": tune.choice([32, 64, 128]),
    "num_layers": tune.choice([1, 2]),
    "model_type": tune.choice(["lstm", "cnn"]),
    "max_epochs": 15,
}

In [ ]:
# ASHA scheduler: aggressively prunes low-performing trials using intermediate `val_loss`
# reports, freeing their fractional GPU slot for new trial configurations.
scheduler = ASHAScheduler(
    metric="val_loss",
    mode="min",
    max_t=search_space["max_epochs"],
    grace_period=3,       # let every trial run at least 3 epochs before it can be stopped
    reduction_factor=2,   # keep the top half of trials at each rung
)

In [ ]:
# Bundle the shared dataset into the trainable, and request FRACTIONAL GPU resources per trial.
# cpu=2, gpu=0.33 lets three trials share one physical Colab GPU concurrently -
# the modern (Tuner-API) equivalent of the legacy `resources_per_trial={"cpu": 2, "gpu": 0.33}`.
trainable_with_data = tune.with_parameters(
    train_forecast, X_train=X_train_np, y_train=y_train_np, X_val=X_val_np, y_val=y_val_np,
)
trainable_with_resources = tune.with_resources(
    trainable_with_data,
    resources={"cpu": 2, "gpu": 0.33},
)

tuner = tune.Tuner(
    trainable_with_resources,
    param_space=search_space,
    tune_config=tune.TuneConfig(
        scheduler=scheduler,
        num_samples=12,             # total trial configurations explored
        max_concurrent_trials=3,    # matches 1 GPU / 0.33 GPU-per-trial
    ),
    run_config=train.RunConfig(
        name="raytune_fractional_gpu_forecast",
        storage_path="/tmp/ray_results",
        verbose=1,
    ),
)

## 5. Execution & Analysis

We now launch the tuning run. Watch the trial table: at any moment you should see **up to 3 `RUNNING` trials** sharing the GPU, and trials that are clearly underperforming will move to `TERMINATED` well before `max_epochs`, courtesy of ASHA.

In [ ]:
# Run the HPO search. This call blocks until all trials complete or are pruned.
results = tuner.fit()

In [ ]:
# Extract the best trial and its hyperparameters/metrics.
best_result = results.get_best_result(metric="val_loss", mode="min")

print("Best hyperparameters found:")
for key, val in best_result.config.items():
    print(f"  {key:>12}: {val}")

print(f"\nBest val_loss  : {best_result.metrics['val_loss']:.4f}")
print(f"Best train_loss: {best_result.metrics['train_loss']:.4f}")

In [ ]:
# Plot the validation-loss learning curves for every trial, highlighting the best one.
# Trials pruned early by ASHA simply have shorter curves, visually showing the compute saved.
fig, ax = plt.subplots(figsize=(9, 5))

for result in results:
    df = result.metrics_dataframe
    if df is None or "val_loss" not in df:
        continue
    is_best = result.config == best_result.config
    ax.plot(df["epoch"], df["val_loss"],
            color="crimson" if is_best else "steelblue",
            linewidth=2.5 if is_best else 1.0,
            alpha=1.0 if is_best else 0.4,
            label="Best trial" if is_best else None)

ax.set_xlabel("Epoch")
ax.set_ylabel("Validation loss (MSE, normalized target)")
ax.set_title("ASHA-scheduled trials: validation loss per epoch")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Summarize final performance across all trials as a bar chart, sorted best-to-worst.
result_df = results.get_dataframe()
result_df = result_df.sort_values("val_loss").reset_index(drop=True)

fig, ax = plt.subplots(figsize=(9, 5))
colors = ["crimson" if i == 0 else "steelblue" for i in range(len(result_df))]
ax.bar(range(len(result_df)), result_df["val_loss"], color=colors)
ax.set_xlabel("Trial (sorted by final val_loss)")
ax.set_ylabel("Final validation loss")
ax.set_title("Trial performance summary")
plt.tight_layout()
plt.show()

result_df[["config/model_type", "config/hidden_size", "config/num_layers",
           "config/lr", "config/dropout", "config/batch_size", "val_loss"]].head(10)

In [ ]:
# Clean resource teardown: release the Ray cluster and clear the driver's CUDA cache.
ray.shutdown()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Ray cluster shut down and CUDA cache cleared.")

## Summary

- **cuDF** turned rolling/lag/groupby feature engineering over 500k rows into a sub-second GPU operation, so preprocessing never gates the HPO loop.
- **Fractional GPU allocation** (`gpu=0.33` per trial) let Ray Tune run 3 PyTorch trials concurrently on Colab's single GPU instead of one at a time — roughly a 3x increase in trials explored per hour.
- **ASHA** killed weak trials early, further concentrating GPU time on promising configurations.
- Together, these techniques make serious HPO practical on the single-GPU budgets typical of Colab, a laptop, or a single cloud instance — no multi-GPU cluster required.